# Qwen activation smoke test
Select **Runtime → Change runtime type → GPU** before running. Run cells in order and upload the repository ZIP downloaded from GitHub. Outputs are saved to Google Drive. This notebook checks infrastructure only; it does not collect research data.

## 1. Upload and open the project

In [1]:
from google.colab import files
from pathlib import Path
import os
import zipfile

uploaded = files.upload()
archives = [name for name in uploaded if name.lower().endswith(".zip")]
assert len(archives) == 1, "Upload exactly one repository ZIP."
with zipfile.ZipFile(archives[0]) as archive:
    scripts = [name for name in archive.namelist() if name.endswith("/src/run_model.py")]
    assert len(scripts) == 1, "ZIP must contain exactly one project with src/run_model.py."
    archive.extractall("/content")
project_dir = Path("/content") / Path(scripts[0]).parent.parent
os.chdir(project_dir)
print("Project:", Path.cwd())

Saving ai-safety-capstone.zip to ai-safety-capstone.zip
Project: /content/ai-safety-capstone


## 2. Install dependencies and check the GPU

In [2]:
%pip install -r requirements.txt

import torch
assert torch.cuda.is_available(), "Select a GPU runtime, then rerun this cell."
print("GPU:", torch.cuda.get_device_name(0))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 109.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 117.1 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.33.0
    Uninstalling huggingface_hub-1.33.0:
      Successfully uninstalled huggingface_hub-1.33.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.23.3
    Uninstalling tokenizers-0.23.3:
      Successfully uninstalled tokenizers-0.23.3
  Attempting uninstall: transformers
    Found existing installation: transformers 5.20.1
    Uninstalling transformers-5.20.1:
      Successfully uninstalled transformers-5.20.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of t

## 3. Mount persistent storage and choose a fresh output folder

In [3]:
from google.colab import drive
from datetime import datetime, timezone

drive.mount("/content/drive")
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
run_dir = Path("/content/drive/MyDrive/ai-safety-capstone") / f"smoke_{stamp}"
print("Save this path for reloading after a restart:", run_dir)

Mounted at /content/drive
Save this path for reloading after a restart: /content/drive/MyDrive/ai-safety-capstone/smoke_20261022T150546349901Z


## 4. Run Qwen and extract activations
The first run downloads model weights. Wait for the script to finish before continuing.

In [4]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "src/run_model.py", "--device", "cuda", "--output-dir", str(run_dir)],
    check=True,
)

CompletedProcess(args=['/usr/bin/python3', 'src/run_model.py', '--device', 'cuda', '--output-dir', '/content/drive/MyDrive/ai-safety-capstone/smoke_20261022T150546349901Z'], returncode=0)

## 5. Inspect saved outputs
Expect three float32 arrays of shape `(1, 2048)` and a response answering 42. Inspect the response yourself.

To check persistence later: restart the runtime, mount Drive again, set `run_dir` to the printed saved path, and rerun this cell. It does not need the model loaded.

In [5]:
import json
import numpy as np
from pathlib import Path

run_dir = Path(run_dir)
metadata = json.loads((run_dir / "metadata.json").read_text())
assert set(metadata["activations"]) == {"9", "18", "27"}, "Expected all three default layers."
for layer, info in metadata["activations"].items():
    vector = np.load(run_dir / info["file"], allow_pickle=False)
    assert vector.shape == (1, 2048) == tuple(info["shape"])
    assert vector.dtype == np.float32
    assert np.isfinite(vector).all()
    print(f"Layer {layer}: {vector.shape}, {vector.dtype}, finite")
print("Response:", metadata["response"])
print("Model revision:", metadata["resolved_revision"])
measurements = metadata.get("measurements")
if measurements is not None:
    print("Timing scope:", measurements["scope"])
    print(f"Elapsed: {measurements['elapsed_seconds']:.2f} seconds")
    if measurements["peak_gpu_allocated_bytes"] is not None:
        print(f"Peak GPU allocated: {measurements['peak_gpu_allocated_bytes'] / 1024**3:.3f} GiB")
        print(f"Peak GPU reserved: {measurements['peak_gpu_reserved_bytes'] / 1024**3:.3f} GiB")
    print("Memory scope:", measurements["memory_scope"])
else:
    print("This run did not record runtime or peak GPU memory.")

Layer 9: (1, 2048), float32, finite
Layer 18: (1, 2048), float32, finite
Layer 27: (1, 2048), float32, finite
Response: 42
Model revision: aa8e72537993ba99e69dfaafa59ed015b17504d1
Timing scope: seed setup, tokenizer/model loading (including downloads if needed), prompt processing, activation extraction, generation, and decoding; excludes imports and output saving
Elapsed: 81.70 seconds
Peak GPU allocated: 5.773 GiB
Peak GPU reserved: 5.869 GiB
Memory scope: PyTorch CUDA allocator in this script process; not total GPU usage
